# Cleaning, Profiling and Quality in Every Tool

Notebook for **Chapter 3** of the *ways* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ways/ch03.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
# Setup: the files used in this chapter are downloaded next to the notebook.
import os, urllib.request
os.makedirs("data", exist_ok=True)
for f in ["customers.csv", "orders.csv", "products.csv", "shopkart.db"]:
    if not os.path.exists("data/" + f):
        urllib.request.urlretrieve("https://dadipavan.github.io/Learning/study/ways/data/" + f, "data/" + f)


## 3.1 A cleaning process you can defend

Cleaning is a set of decisions about the truth, so every step should be justified and recorded. Use the same five moves every time.

## 3.2 Duplicates

A duplicate is the same real-world event recorded more than once, often by a retried upload. Exact duplicates (every column equal) are safe to drop. Partial duplicates (same id, different status) need a rule such as "keep the latest". ShopKart has 150 exact duplicates.

In [ ]:
DATA = "data/"
import sqlite3, csv
import pandas as pd

df = pd.read_csv(DATA + "orders.csv")
print("pandas :", df.duplicated().sum(), "duplicates ->", len(df.drop_duplicates()), "rows")

con = sqlite3.connect(DATA + "shopkart.db")
n_all = con.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
n_distinct = con.execute("SELECT COUNT(*) FROM (SELECT DISTINCT * FROM orders)").fetchone()[0]
print("sql    :", n_all - n_distinct, "duplicates ->", n_distinct, "rows")

seen, kept, n_read = set(), [], 0
with open(DATA + "orders.csv", newline="") as f:
    next(f)                       # skip the header line
    for row in csv.reader(f):
        n_read += 1
        t = tuple(row)
        if t not in seen:
            seen.add(t)
            kept.append(t)
print("python :", n_read - len(kept), "duplicates ->", len(kept), "rows")

pandas : 150 duplicates -> 20000 rows
sql    : 150 duplicates -> 20000 rows
python : 150 duplicates -> 20000 rows


## 3.3 Inconsistent text: the city spellings

Text typed by people or collected from several systems drifts: case, spaces, old names (Bombay, Madras), aliases (New Delhi). The process: list the distinct values with counts, build a mapping to canonical names, apply it, and check the output has only canonical values plus missing.

In [ ]:
DATA = "data/"
import pandas as pd
df = pd.read_csv(DATA + "orders.csv")
print(df.city.value_counts(dropna=False).to_string())

city
Bengaluru     5226
Mumbai        4332
Delhi         3985
Pune          2321
Chennai       2266
MUMBAI         176
mumbai         172
Bangalore      160
Bombay         155
bangalore      153
delhi          150
New Delhi      145
Madras         142
BENGALURU      138
chennai        138
PUNE           125
DELHI          124
pune           122
NaN            120


In [ ]:
DATA = "data/"
import pandas as pd
import sqlite3
df = pd.read_csv(DATA + "orders.csv")
alias = {"bangalore": "Bengaluru", "bengaluru": "Bengaluru", "mumbai": "Mumbai", "bombay": "Mumbai",
         "delhi": "Delhi", "new delhi": "Delhi", "chennai": "Chennai", "madras": "Chennai", "pune": "Pune"}

# pandas
fixed = df.city.str.strip().str.lower().map(alias)
print("pandas:", fixed.value_counts(dropna=False).to_dict())

# SQL: same idea with lower, trim and CASE
con = sqlite3.connect(DATA + "shopkart.db")
q = """
SELECT CASE LOWER(TRIM(city))
         WHEN 'bangalore' THEN 'Bengaluru' WHEN 'bengaluru' THEN 'Bengaluru'
         WHEN 'mumbai' THEN 'Mumbai' WHEN 'bombay' THEN 'Mumbai'
         WHEN 'delhi' THEN 'Delhi' WHEN 'new delhi' THEN 'Delhi'
         WHEN 'chennai' THEN 'Chennai' WHEN 'madras' THEN 'Chennai'
         WHEN 'pune' THEN 'Pune' END AS city_clean, COUNT(*) AS n
FROM orders GROUP BY 1 ORDER BY n DESC"""
print("sql   :", {str(k): v for k, v in con.execute(q).fetchall()})

pandas: {'Bengaluru': 5677, 'Mumbai': 4835, 'Delhi': 4404, 'Pune': 2568, 'Chennai': 2546, nan: 120}
sql   : {'Bengaluru': 5677, 'Mumbai': 4835, 'Delhi': 4404, 'Pune': 2568, 'Chennai': 2546, 'None': 120}


## 3.4 Missing values

A missing value is not one problem but several. The right action depends on *why* it is missing.

In [ ]:
DATA = "data/"
import sqlite3, pandas as pd
o = pd.read_csv(DATA + "orders.csv").drop_duplicates()
p = pd.read_csv(DATA + "products.csv")
missing = o.unit_price.isna()
filled = o.unit_price.fillna(o["product"].map(p.set_index("product")["list_price"]))
print("pandas: rows filled", missing.sum(), "| still missing", filled.isna().sum(),
      "| gross value of those rows", round((o.quantity * filled)[missing & (o.quantity <= 20)].sum()))

con = sqlite3.connect(DATA + "shopkart.db")
r = con.execute("""
  SELECT COUNT(*), SUM(COALESCE(o.unit_price, p.list_price) IS NULL), ROUND(SUM(o.quantity * p.list_price), 0)
  FROM (SELECT DISTINCT * FROM orders) o JOIN products p ON p.product = o.product
  WHERE o.unit_price IS NULL AND o.quantity <= 20""").fetchone()
print("sql   : rows filled", r[0], "| still missing", r[1], "| gross value of those rows", int(r[2]))

pandas: rows filled 90 | still missing 0 | gross value of those rows 107500
sql   : rows filled 90 | still missing 0 | gross value of those rows 107500


In [ ]:
DATA = "data/"
import pandas as pd
c = pd.read_csv(DATA + "customers.csv")
print("missing ages:", c.age.isna().sum(), "of", len(c))
print("mean", round(c.age.mean(), 2), "| median", c.age.median())
print(c.groupby("segment").age.median().to_dict())
by_seg = c.age.fillna(c.groupby("segment").age.transform("median"))
print("after group-median fill: missing", by_seg.isna().sum(), "| overall mean", round(by_seg.mean(), 2))

missing ages: 146 of 4000
mean 32.69 | median 32.0
{'Premium': 34.0, 'Regular': 34.0, 'Student': 22.0}
after group-median fill: missing 0 | overall mean 32.68


## 3.5 Outliers: errors versus extremes

An outlier is a value far from the rest. Two different things hide behind the word: **errors** (a quantity of 99 where the maximum possible basket is a handful; a delivery time of minus 12 minutes) and **genuine extremes** (a legitimate rural delivery that took 3 hours). Errors are fixed or removed; extremes are kept, perhaps capped for a model. Only domain knowledge separates them.

In [ ]:
DATA = "data/"
import pandas as pd
o = pd.read_csv(DATA + "orders.csv").drop_duplicates()
d = o.delivery_minutes.dropna()
q1, q3 = d.quantile([.25, .75])
lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
z = (d - d.mean()) / d.std()
print(f"Q1={q1} Q3={q3} IQR fences: {lo:.1f} to {hi:.1f}")
print("IQR flags :", ((d < lo) | (d > hi)).sum(), "| of which above 300:", (d > 300).sum(), "| negative:", (d < 0).sum())
print("z>3 flags :", (z.abs() > 3).sum())
print("domain rule (outside 5..300):", ((d < 5) | (d > 300)).sum())
print("mean with errors", round(d.mean(), 2), "| median", d.median(), "| mean without errors", round(d[d.between(5, 300)].mean(), 2))

Q1=34.0 Q3=46.0 IQR fences: 16.0 to 64.0
IQR flags : 232 | of which above 300: 12 | negative: 8
z>3 flags : 12
domain rule (outside 5..300): 20
mean with errors 41.12 | median 39.0 | mean without errors 40.22


## 3.6 Types, dates and units

A type error is a silent bug: text that looks like a number sorts as text ("10" before "9"), a date read as text cannot be subtracted, and an id read as a float turns 1001 into 1001.0, which then fails to match a text id. The checklist: ids as integers or strings (never floats), dates as dates, money as decimals with one currency, flags as booleans, categories as a controlled set.

In [ ]:
DATA = "data/"
import pandas as pd
o = pd.read_csv(DATA + "orders.csv")
print(o.dtypes.to_string())
o["order_date"] = pd.to_datetime(o["order_date"])
print("days in range:", (o.order_date.max() - o.order_date.min()).days + 1)
print("text sort:", sorted(["10", "9", "100"]), "| number sort:", sorted([10, 9, 100]))
o["month"] = o.order_date.dt.to_period("M").astype(str)
print(o.month.nunique(), "months; first", o.month.min())

order_id              int64
customer_id           int64
order_date              str
order_hour            int64
city                    str
channel                 str
category                str
product                 str
quantity              int64
unit_price          float64
discount            float64
status                  str
promised_minutes    float64
delivery_minutes    float64
days in range: 366
text sort: ['10', '100', '9'] | number sort: [9, 10, 100]
12 months; first 2024-01


## 3.7 What cleaning did to the answers

The point of a log is to show the effect of each rule. We apply the rules one after another and track the headline metrics, in pandas with the exact same order as `prep.py`.

In [ ]:
DATA = "data/"
import pandas as pd
o = pd.read_csv(DATA + "orders.csv")
p = pd.read_csv(DATA + "products.csv")
price = lambda d: d.unit_price.fillna(d["product"].map(p.set_index("product")["list_price"]))
rev = lambda d: (d.quantity * price(d) * (1 - d.discount))[d.status == "delivered"].sum()
log = [("raw file", len(o), rev(o))]
o = o.drop_duplicates();                 log.append(("drop 150 exact duplicates", len(o), rev(o)))
o = o[o.quantity <= 20];                 log.append(("drop quantity > 20", len(o), rev(o)))
raw_missing_price = o.unit_price.isna().sum()
rows = pd.DataFrame(log, columns=["step", "rows", "delivered_revenue"])
rows["change"] = rows.delivered_revenue.diff().round(0)
print(rows.round(0).to_string(index=False))
nofill = (o.quantity * o.unit_price * (1 - o.discount))[o.status == "delivered"].sum()
print("revenue if missing prices are NOT filled:", round(nofill), "| fill adds", round(rev(o) - nofill))

step  rows  delivered_revenue    change
                 raw file 20150         17674094.0       NaN
drop 150 exact duplicates 20000         17550598.0 -123496.0
       drop quantity > 20 19995         17349916.0 -200683.0
revenue if missing prices are NOT filled: 17274362 | fill adds 75553


## 3.8 Which tool for which cleaning job

## Chapter recap

## Practice set

Problem 3.9.1 How many exact duplicate rows does orders.csv contain, and how many rows remain after dropping them?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ways/ch03.html